# Ingest iSnobal Historical SWE

One-time backfill of the iSnobal `specific_mass` (SWE) GeoTIFFs in `s3://m3w-transfer/nrcs/shared/` into one IceChunk repository per domain (`isnobal_<domain>`).

This calls the `ingest_gridded_data` flow directly, so the ingest code is the same as the scheduled flow's. Prefect runs in-process on a temporary local server; no deployment or Prefect API is needed.

- Water years run in order within a domain: the flow only appends, so an earlier year after a later one would leave the time axis unsorted.
- Domains run in parallel; each has its own repository.
- A failed water year stops its domain. Re-running is safe, since each stage of the flow skips steps it already has.

Credentials are read from the environment:
- `M3WORKS_AWS_ACCESS_KEY_ID`, `M3WORKS_AWS_SECRET_ACCESS_KEY`: the source bucket.
- `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (and `AWS_REGION`): the destination IceChunk bucket.

Install the flow's dependencies, pinned as in `prefect-workflows/Dockerfile.prefect-teehr`. Restart the kernel afterwards.

In [ ]:
%pip install "icechunk~=2.1.2" "xarray<2026.9" "topozarr~=0.1.3" "virtual-tiff~=0.5.0" "xproj~=0.2.1" "rioxarray~=0.23.0" "prefect==3.4.24" "fastapi<0.120"

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Prefect logs every task; keep the notebook output to warnings and the progress lines below
os.environ.setdefault("PREFECT_LOGGING_LEVEL", "WARNING")

# The flow imports from both of these, as when Prefect runs it from its entrypoint
PREFECT_WORKFLOWS = Path.cwd().parents[2] / "prefect-workflows"
sys.path[:0] = [str(PREFECT_WORKFLOWS), str(PREFECT_WORKFLOWS / "workflows" / "ingests")]

import icechunk as ic
import pandas as pd
import xarray as xr
from prefect import flow
from ingest_gridded_data import ingest_gridded_data
from utils import grid_utils as gu
from workflows.models.ingest_gridded_data_input import IngestGriddedDataInput

Set the destination, then check the credentials are present.

In [ ]:
DEST_BUCKET = "CHANGE_ME"
BASE_PREFIX = "CHANGE_ME"

for name in ("M3WORKS_AWS_ACCESS_KEY_ID", "M3WORKS_AWS_SECRET_ACCESS_KEY", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    print(f"{name}: {'set' if os.environ.get(name) else 'MISSING'}")

First and last water year of each domain in the bucket (surveyed 2026-09-30).

In [ ]:
DOMAINS = {
    "boise": (1980, 2024),
    "flathead": (1980, 2024),
    "gunnison_east_taylor": (1991, 2024),
    "north_fork_clearwater": (2014, 2024),
    "san_juan_mountains": (1980, 2024),
    "south_san_juans": (1991, 2024),
    "uintas_extended": (1980, 2024),
    "upper_rio_grande": (1991, 2024),
    "willamette_extended": (1980, 2024),
}

In [ ]:
def last_ingested_step(domain: str) -> pd.Timestamp | None:
    """Latest step in the domain's /pyramids/0, the flow's last stage; None if there is none yet."""
    storage = gu.build_icechunk_s3_storage(DEST_BUCKET, f"{BASE_PREFIX}/isnobal_{domain}", from_env=True)
    if not ic.Repository.exists(storage):
        return None
    store = ic.Repository.open(storage).readonly_session("main").store
    if not gu.group_contains_data(store, "/pyramids/0"):
        return None
    return pd.Timestamp(xr.open_zarr(store, group="/pyramids/0", consolidated=False).time.values.max())


def backfill_domain(domain: str, first_wy: int, last_wy: int) -> str:
    """Ingest each water year of a domain in order, from the first not fully ingested, stopping at the first failure."""
    last = last_ingested_step(domain)
    if last is not None:
        # Resume at the water year holding the next day; re-running it completes any stage left unfinished
        next_day = last + pd.Timedelta(days=1)
        first_wy = max(first_wy, next_day.year + (next_day.month >= 10))
    for wy in range(first_wy, last_wy + 1):
        args = IngestGriddedDataInput(
            source={"type": "isnobal", "domain": domain},
            parser_type="tiff",
            x_dim="x",
            y_dim="y",
            start_dt=f"{wy - 1}-10-01T00:00:00",
            end_dt=f"{wy}-09-30T23:00:00",
            dest_bucket=DEST_BUCKET,
            base_prefix=BASE_PREFIX,
            factors=[1, 2],
            # A whole shard per pyramid batch, so shards aren't rewritten
            time_batch_size=30,
            # Fail rather than skip a file that exists but can't be read; a skipped day can't be filled later
            ignore_unreadable_file=False,
        )
        try:
            ingest_gridded_data(args)
        except Exception as e:
            return f"{domain}: stopped at WY{wy} ({type(e).__name__}: {e})"
        print(f"{domain} WY{wy}: done")
    return f"{domain}: done"


# Start the temporary Prefect server once; flows started together in threads would each
# try to start it, and their database migrations collide
@flow
def start_prefect_server():
    pass


start_prefect_server()

Try one domain for two water years first, then check the repository (time axis monotonic, no gaps) before running the rest.

In [ ]:
backfill_domain("boise", 1980, 1981)

In [ ]:
with ThreadPoolExecutor(max_workers=len(DOMAINS)) as pool:
    for result in pool.map(lambda item: backfill_domain(item[0], *item[1]), DOMAINS.items()):
        print(result)